# Phase 2 (Extended): Multi-Model Ensemble Classifier

**Objective:** Strengthen the Phase 2 baseline beyond a single Random Forest by combining multiple model families into a stacked ensemble. This is the production pattern used in financial ML for fraud detection, credit risk, and anomaly classification.

**Models combined:**
1. **Random Forest** — bagging-based ensemble, captures non-linear interactions.
2. **XGBoost** — sequential gradient boosting, corrects errors of weak learners.
3. **LightGBM** — leaf-wise gradient boosting, faster and stronger on imbalanced data.
4. **Logistic Regression** — meta-learner for stacking; learns optimal weighting of base models.

**Combination strategies tested:**
- Soft Voting (probability averaging)
- Stacking (LR meta-learner over RF / XGBoost / LightGBM out-of-fold predictions)

**Connection to other phases:** This stacked ensemble *replaces* the single-RF baseline as Phase 2. Phase 3 GAN-synthesised samples augment the training set for the *entire* ensemble. Phase 4 HuggingFace NLP runs as a separate parallel pipeline for unstructured text.

In [ ]:
# Install if needed:
# !pip install xgboost lightgbm scikit-learn matplotlib seaborn -q

import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print(f'XGBoost version : {xgb.__version__}')
print(f'LightGBM version: {lgb.__version__}')

## 1. Load & Preprocess NSL-KDD

In [ ]:
COLUMNS = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
    'num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
    'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
    'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
]

try:
    df = pd.read_csv('../data/KDDTrain+.txt', names=COLUMNS)
    print(f'Loaded NSL-KDD: {df.shape}')
except FileNotFoundError:
    print('Dataset not found — generating demo data for illustration')
    n = 5000
    df = pd.DataFrame(np.random.randn(n, 38), columns=COLUMNS[:38])
    df['label'] = np.random.choice(
        ['normal','neptune','smurf','back','satan','ipsweep','portsweep',
         'guess_passwd','buffer_overflow','rootkit'],
        p=[0.55,0.15,0.08,0.05,0.05,0.04,0.03,0.03,0.01,0.01], size=n
    )
    df['difficulty'] = 0

# Map fine-grained labels into 5 attack categories
CATEGORY_MAP = {
    'normal': 'Normal',
    'neptune': 'DoS', 'back': 'DoS', 'land': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'mailbomb': 'DoS', 'apache2': 'DoS',
    'processtable': 'DoS', 'udpstorm': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    'mscan': 'Probe', 'saint': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R'
}
df['category'] = df['label'].map(lambda x: CATEGORY_MAP.get(x, 'Other'))
print('\nClass distribution:')
print(df['category'].value_counts())

In [ ]:
# Encode categorical features
for col in ['protocol_type', 'service', 'flag']:
    if col in df.columns and df[col].dtype == 'object':
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Build feature matrix
feature_cols = [c for c in COLUMNS[:41] if c not in ['label', 'difficulty']]
feature_cols = [c for c in feature_cols if c in df.columns]

scaler = MinMaxScaler()
X = scaler.fit_transform(df[feature_cols].fillna(0))

# Encode target
y_encoder = LabelEncoder()
y = y_encoder.fit_transform(df['category'])
class_names = y_encoder.classes_

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Classes: {list(class_names)}')

## 2. Base Models (Individual Performance)

In [ ]:
# === Random Forest (bagging baseline) ===
rf = RandomForestClassifier(
    n_estimators=200, max_depth=20,
    n_jobs=-1, random_state=42,
    class_weight='balanced'
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print(f'Random Forest accuracy: {accuracy_score(y_test, rf_pred):.4f}')
print(f'Random Forest macro-F1: {f1_score(y_test, rf_pred, average="macro"):.4f}')

In [ ]:
# === XGBoost (sequential gradient boosting) ===
xgb_clf = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1,
    random_state=42
)
xgb_clf.fit(X_train, y_train)
xgb_pred = xgb_clf.predict(X_test)
print(f'XGBoost accuracy: {accuracy_score(y_test, xgb_pred):.4f}')
print(f'XGBoost macro-F1: {f1_score(y_test, xgb_pred, average="macro"):.4f}')

In [ ]:
# === LightGBM (leaf-wise gradient boosting) ===
lgb_clf = lgb.LGBMClassifier(
    n_estimators=300,
    num_leaves=63,
    learning_rate=0.1,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
    verbose=-1
)
lgb_clf.fit(X_train, y_train)
lgb_pred = lgb_clf.predict(X_test)
print(f'LightGBM accuracy: {accuracy_score(y_test, lgb_pred):.4f}')
print(f'LightGBM macro-F1: {f1_score(y_test, lgb_pred, average="macro"):.4f}')

## 3. Soft Voting Ensemble
Average probability predictions across all three base models.

In [ ]:
voting = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('xgb', xgb_clf),
        ('lgb', lgb_clf)
    ],
    voting='soft',
    n_jobs=-1
)
voting.fit(X_train, y_train)
vote_pred = voting.predict(X_test)
print(f'Voting ensemble accuracy: {accuracy_score(y_test, vote_pred):.4f}')
print(f'Voting ensemble macro-F1: {f1_score(y_test, vote_pred, average="macro"):.4f}')
print('\nClassification report:')
print(classification_report(y_test, vote_pred, target_names=class_names, zero_division=0))

## 4. Stacking Ensemble
Logistic Regression learns the optimal weighting of base-model predictions using out-of-fold cross-validation.

In [ ]:
stacking = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight='balanced')),
        ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1,
                                  tree_method='hist', n_jobs=-1, random_state=42, eval_metric='mlogloss')),
        ('lgb', lgb.LGBMClassifier(n_estimators=200, num_leaves=63, n_jobs=-1,
                                    random_state=42, verbose=-1, class_weight='balanced'))
    ],
    final_estimator=LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)

print('Training stacked ensemble (5-fold CV) ...')
stacking.fit(X_train, y_train)
stack_pred = stacking.predict(X_test)

print(f'\nStacked ensemble accuracy: {accuracy_score(y_test, stack_pred):.4f}')
print(f'Stacked ensemble macro-F1: {f1_score(y_test, stack_pred, average="macro"):.4f}')
print('\nClassification report:')
print(classification_report(y_test, stack_pred, target_names=class_names, zero_division=0))

## 5. Comparison & Visualisation

In [ ]:
results = pd.DataFrame({
    'Model':    ['Random Forest', 'XGBoost', 'LightGBM', 'Voting Ensemble', 'Stacking Ensemble'],
    'Accuracy': [accuracy_score(y_test, p) for p in [rf_pred, xgb_pred, lgb_pred, vote_pred, stack_pred]],
    'Macro-F1': [f1_score(y_test, p, average='macro') for p in [rf_pred, xgb_pred, lgb_pred, vote_pred, stack_pred]]
})
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#5DCAA5', '#7F77DD', '#EF9F27', '#378ADD', '#D85A30']
axes[0].barh(results['Model'], results['Accuracy'], color=colors)
axes[0].set_xlim(0.85, 1.0)
axes[0].set_title('Test Accuracy', fontsize=13)
axes[0].set_xlabel('Accuracy')
for i, v in enumerate(results['Accuracy']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10)

axes[1].barh(results['Model'], results['Macro-F1'], color=colors)
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title('Macro-F1 Score (rare-class sensitive)', fontsize=13)
axes[1].set_xlabel('Macro-F1')
for i, v in enumerate(results['Macro-F1']):
    axes[1].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../src/results/ensemble_comparison.png', dpi=150)
plt.show()
print('Saved: src/results/ensemble_comparison.png')

In [ ]:
# Confusion matrix for best model (stacking)
cm = confusion_matrix(y_test, stack_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Stacking Ensemble (Phase 2 Final)', fontsize=13)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('../src/results/stacking_confusion_matrix.png', dpi=150)
plt.show()

## 6. Feature Importance Across Models
Cross-checking which features each model relies on — a sanity check on what the ensemble has learned.

In [ ]:
fi = pd.DataFrame({
    'Feature':  feature_cols,
    'RF':       rf.feature_importances_,
    'XGBoost':  xgb_clf.feature_importances_,
    'LightGBM': lgb_clf.feature_importances_ / lgb_clf.feature_importances_.sum()
})
fi['Avg'] = fi[['RF', 'XGBoost', 'LightGBM']].mean(axis=1)
top = fi.nlargest(15, 'Avg').sort_values('Avg')

fig, ax = plt.subplots(figsize=(11, 7))
x = np.arange(len(top))
w = 0.27
ax.barh(x - w, top['RF'],       w, label='Random Forest', color='#5DCAA5')
ax.barh(x,     top['XGBoost'],  w, label='XGBoost',       color='#7F77DD')
ax.barh(x + w, top['LightGBM'], w, label='LightGBM',      color='#EF9F27')
ax.set_yticks(x); ax.set_yticklabels(top['Feature'])
ax.set_xlabel('Normalised Feature Importance')
ax.set_title('Top-15 Features — Cross-Model Importance Comparison', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('../src/results/feature_importance_ensemble.png', dpi=150)
plt.show()

## 7. Save Best Model for Phase 3 Integration
The stacked ensemble becomes the *target* of Phase 3 GAN augmentation — synthetic R2L/U2R samples will be added to the training set, and the entire ensemble retrained.

In [ ]:
import joblib
import os
os.makedirs('../src/models', exist_ok=True)
joblib.dump(stacking,  '../src/models/phase2_stacking_ensemble.pkl')
joblib.dump(scaler,    '../src/models/phase2_feature_scaler.pkl')
joblib.dump(y_encoder, '../src/models/phase2_label_encoder.pkl')

print('Saved:')
print('  src/models/phase2_stacking_ensemble.pkl')
print('  src/models/phase2_feature_scaler.pkl')
print('  src/models/phase2_label_encoder.pkl')
print('\nPhase 2 complete. The stacked ensemble (RF + XGBoost + LightGBM)'
      ' is now ready for GAN-augmented retraining in Phase 3.')